# Solar Filament Segmentation 2026: end-to-end pipeline

This notebook reproduces the submitted predictions from the files in `weights/`: five EfficientNet-B3 U-Nets (one per cross-validation fold) and the decision-layer scorer. The method is described in `README.md` and `report/main.pdf`. Re-training is optional and listed at the end.

Requirements: `pip install -r requirements.txt`, and the Kaggle data unzipped into `data/raw/MAGFiLO_1.0_Kaggle_2026/`. A CUDA GPU is needed for the U-Nets; the decision layer runs on CPU.

In [ ]:
import sys, os, glob, json, pickle
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')
import numpy as np, pandas as pd, cv2, torch, matplotlib.pyplot as plt
from common import RAW, load_image
from predict import load_models, predict_prob
from decision import process_image, select, _score, FEATURES
PARAMS = json.load(open('weights/params.json'))
WEIGHTS = sorted(glob.glob('weights/effb3d_f*.pt'))
with open(PARAMS['scorer'], 'rb') as fh:
    SCORER = pickle.load(fh)
TAU = PARAMS['tau']
print(PARAMS, WEIGHTS, 'cuda:', torch.cuda.is_available())

## 1. Data and targets
Training annotations are COCO polygons from 1 to 3 annotators per image. `src/prepare.py` rasterises them into per-annotator instance maps and three consensus targets: the soft filament mask (fraction of annotators), a normalised distance-to-boundary map and the contact line between touching filaments. Folds are grouped by observation date.

In [ ]:
if not os.path.exists('data/masks/touch'):
    !python src/prepare.py
splits = pd.read_csv('data/splits.csv'); print(splits.fold.value_counts().sort_index().to_dict(), 'files per fold')
stem = splits[splits.fold == 0].stem.iloc[3]
img = load_image(f'{RAW}/train/train_images/{stem}.jpeg')
soft = cv2.imread(f'data/masks/soft/{stem}.png', 0)
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(img, cmap='gray'); ax[0].set_title(stem); ax[1].imshow(soft, cmap='magma'); ax[1].set_title('consensus mask')
for a in ax: a.axis('off')
plt.show()

## 2. Dense prediction
Each model predicts the whole 2048 x 2048 frame with flip test-time augmentation; the five fold models are averaged. Output channels: filament probability, distance map, contact line.

In [ ]:
test_paths = sorted(glob.glob(f'{RAW}/test/test_images/*.jpeg'))
models = load_models(WEIGHTS, 'cuda')
example = os.path.basename(test_paths[0])[:-5]
out = predict_prob(models, load_image(test_paths[0]), 'cuda', tta=True)
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(load_image(test_paths[0]), cmap='gray'); ax[0].set_title(example); ax[1].imshow(out[0], cmap='magma'); ax[1].set_title('5-fold mean filament probability')
for a in ax: a.axis('off')
plt.show()

## 3. Decision layer: component tree, scorer, exact selection
The probability map is thresholded at 16 levels; the nested connected components form a forest of candidates. A gradient-boosted hurdle model predicts each candidate's expected matched IoU against a random annotator, and a dynamic programme selects the disjoint set that maximises the sum of (score - tau). Idea credited to the public notebook by _.tushhaaarrrr (see README).

In [ ]:
os.makedirs('runs/maps/test_effb3d', exist_ok=True)
np.save(f'runs/maps/test_effb3d/{example}.npy', (out * 255).round().astype(np.uint8))
r = process_image(example, f'runs/maps/test_effb3d/{example}.npy', test_paths[0], keep_labels=True)
nd = r['nodes']; u = _score(SCORER['model'], nd[FEATURES].to_numpy(np.float32))
chosen = select(nd.parent.to_numpy(), u - TAU)
inst = np.zeros((2048, 2048), np.int32)
for k, v in enumerate(chosen, 1):
    inst[r['labels'][int(nd.level_idx.iloc[v])] == int(nd.lab_id.iloc[v])] = k
print(f'{len(nd)} candidates, {len(chosen)} selected')
fig, ax = plt.subplots(1, 2, figsize=(14, 7)); ax[0].imshow(out[0] > 0.6, cmap='gray'); ax[0].set_title('plain rule p > 0.6'); ax[1].imshow(np.ma.masked_equal(inst, 0), cmap='nipy_spectral'); ax[1].set_title(f'decision layer, tau = {TAU}')
for a in ax: a.axis('off')
plt.show()

## 4. Validation summary (out-of-fold)
Pooled PQ as the host computes it, per fold. These numbers come from `src/decision.py fit` and `scripts/make_decision_table.py`; recomputing them needs the out-of-fold maps (see section 6).

In [ ]:
summary = json.load(open('report/decision_summary.json'))
pd.DataFrame({k: v for k, v in summary.items() if k != 'tau'}).T[['baseline', 'fixed', 'decision', 'oracle']].round(4)

## 5. Test predictions and submission file
All 180 test frames: cache the 5-fold mean maps, then run the decision layer. `src/check_submission.py` verifies RLE decodability, unique ids and that no two masks of an image overlap.

In [ ]:
!python src/predict_maps.py test --weights {' '.join(WEIGHTS)} --out runs/maps/test_effb3d
!python src/decision.py predict --maps runs/maps/test_effb3d --model {PARAMS['scorer']} --tau {TAU} --out submission.csv
!python src/check_submission.py submission.csv
pd.read_csv('submission.csv').head()

## 6. (Optional) Re-train and re-fit
```
python src/prepare.py
scripts/train_folds.sh efficientnet-b3 effb3d "0 1 2 3 4" --epochs 50 --crop 640 --bs 6 --eval_every 10
for k in 0 1 2 3 4; do python src/export_weights.py runs/effb3d_f$k weights/effb3d_f$k.pt; done
python src/predict_maps.py oof --weights weights/effb3d_f*.pt --out runs/maps/oof_effb3d
python src/decision.py bank --maps runs/maps/oof_effb3d --out runs/decision/bank_effb3d.pkl
python src/decision.py fit --bank runs/decision/bank_effb3d.pkl --out weights/decision_scorer.pkl
python scripts/make_decision_table.py 0.26
```